# Exploring mypy incremental cache and follow-imports modes

> L4 investigation: how mypy decides to re-type-check a file, what the cache does (and doesn't) catch, how `--follow-imports` reshapes cross-package analysis, and what changes when the Python version target moves.

Two flags drive most of mypy's behavior on multi-file projects: the incremental cache (`.mypy_cache/`) and `--follow-imports=…`. A third axis — `python_version` / `python_executable` — interacts with both. This notebook walks each one with a small reproducible setup, then shows what to expect when you mix them.

Setup: a project with two subpackages, one fully annotated and one intentionally untyped, plus a third-party import. The point is to make the flag behavior observable, not to demo a real project.

## Step 1 — the incremental cache, cold vs warm

mypy writes per-file fingerprints into `.mypy_cache/`. On the second run, files whose fingerprint didn't change are skipped. The `incremental` flag defaults to `True`; setting it to `False` forces a full re-check.

In [ ]:
# Run:  rm -rf .mypy_cache
# Run:  time mypy src/                # cold cache
# Run:  time mypy src/                # warm cache
# Run:  time mypy --no-incremental src/
#
# The warm-cache run drops a large fraction of the time. The
# --no-incremental run matches the cold-cache time — confirming that
# the speedup comes from the cache, not from mypy getting "smarter".
#
# What the cache stores per file: the file's hash, the AST summary,
# the inferred and declared types of top-level definitions, and the
# dependency set (which other files this one's results depend on).
# mtime is NOT used — fingerprints are content-based, so a `touch`
# of an unchanged file does not invalidate the cache.

## Step 2 — what invalidates a cache entry

The cache is keyed on a fingerprint that mixes file content with several "context" hashes. Touch a relevant config knob and the file gets re-checked even if its bytes are byte-identical.

In [ ]:
# Run:  mypy src/            # warm
# Run:  mypy --strict src/   # forces a full re-check, even with no edits
#
# Why: the per-file fingerprint is hashed together with the resolved
# mypy options. Change a flag, change the fingerprint space, invalidate
# everything.
#
# Other inputs that fold into the fingerprint:
#   - the mypy version (upgrading mypy wipes the cache on first run)
#   - the dependency file's fingerprints (a transitive import changed
#     => every importer gets re-checked)
#   - the typeshed/stdlib stubs being used
#   - the plugin set in [tool.mypy].plugins
#
# Practical consequence: in CI, mounting .mypy_cache/ across jobs is
# unsafe across mypy version changes or [tool.mypy] config changes.
# Key the cache by mypy version + a hash of the config block instead.

## Step 3 — `--follow-imports` shapes how non-target files are handled

`--follow-imports` controls what mypy does when it encounters an import that points *outside* the files you asked it to check — the "silent" / "skip" / "error" modes documented in the mypy running-mypy page.

The four values worth knowing:

- `normal` (default) — follow into a package only if it has a `py.typed` marker or you passed it explicitly on the command line.
- `silent` — same as `normal` but suppresses "note: Skipping analyzing" messages. Useful when you want the check to run quietly on a heterogeneous codebase.
- `skip` — never follow; just treat the imported module as `Any`. Fastest, but loses downstream type information.
- `error` — treat non-followed imports as errors. Good for enforcing that every imported package is fully type-checked.

In [ ]:
# Tree:
#   src/
#     pkg_a/        (fully annotated, has py.typed)
#       __init__.py
#     pkg_b/        (untyped, no py.typed)
#       __init__.py
#     app.py        # imports from both
#
# Run:  mypy src/app.py
#   mypy follows into pkg_a (py.typed present) and uses its real types.
#   mypy skips pkg_b (no py.typed) and treats its exports as Any.
#   app.py still gets a full check — its own annotations are honored.
#
# Run:  mypy --follow-imports=silent src/app.py
#   Same behavior as `normal` but no "Skipping analyzing" notes.
#
# Run:  mypy --follow-imports=error src/app.py
#   Reports pkg_b's import as an error: "Skipped module ... has no
#   py.typed marker". This is the gate to use when you want to force
#   every imported package to be type-checked, not silently Any'd.
#
# Run:  mypy --follow-imports=skip src/app.py
#   pkg_a and pkg_b are both treated as Any. Useful when pkg_a is in
#   flux and you only want to check app.py's own surface.

## Step 4 — `py.typed` is the gate, not the import

The `py.typed` marker file is what flips a package from "implicitly Any" to "eligible for following". PEP 561 specifies that a package with `py.typed` at its root advertises full type support; mypy respects this for `--follow-imports=normal|silent`.

Without `py.typed`, the package is treated as untyped even if it ships individual `.pyi` stubs alongside some of its modules. The two signals are not the same.

In [ ]:
# To see whether an installed package is followed or skipped:
#   python -c "import somepkg, os; print(os.path.exists(os.path.join(os.path.dirname(somepkg.__file__), 'py.typed')))"
#
# `True`  => mypy will follow into it under --follow-imports=normal
# `False` => mypy will skip it (Any) unless you force it via
#            --follow-imports=error or a config override.
#
# Library authors who want their users to get real types must ship
# py.typed; type comments inside the .py are not enough to be
# followed by mypy on the consumer side.

## Step 5 — `python_version` shifts the type universe

`python_version = "X.Y"` (or `python_executable = "..."`) changes which stubs mypy uses. A clean re-check is required when this changes — the cache is version-keyed, but stale entries from a different target can still confuse the dependency graph until the next run completes.

Two things to watch:

- Some `sys.version_info`-gated branches only narrow under the right target. Code that passes under `python_version = "3.10"` may surface `int | None` issues under `"3.12"` because the inferred type of certain stdlib calls changed.
- New stdlib stubs are added each release. An import that was a wildcard `Any` under 3.9 may resolve to a concrete class under 3.12 — which can cascade into new errors in call sites that previously got implicit Any.

In [ ]:
# pyproject.toml — example: the same code, two python_version targets
# config = """
# [tool.mypy]
# python_version = "3.10"
# """
#
# config = """
# [tool.mypy]
# python_version = "3.12"
# """
#
# Switching between them and re-running mypy is a useful smoke test
# for "will this code still type-check on the new interpreter?".
# CI: run the type check once per Python version in your supported
# matrix — different targets exercise different stdlib stubs.

## Step 6 — `MYPYPATH` and where mypy looks for imports

MYPYPATH extends the import search path the same way PYTHONPATH does. When mypy can't find a module, the result is "Skipping analyzing" plus `Any`, not an error — which masks real missing-dependency bugs if you aren't watching.

In [ ]:
# Run:  MYPYPATH=./stubs mypy src/app.py
#
# MYPYPATH takes precedence over installed packages for *type* lookup,
# but not for *runtime* import resolution. A common pattern is to
# ship first-party `.pyi` stubs under a top-level `stubs/` directory
# and add it to MYPYPATH in the type-check step, without putting it
# on PYTHONPATH for the actual app process.
#
# If a module shows up as Any in the output and you expected a real
# type, run with --show-error-context and --warn-unused-ignores to
# surface the "library stubs not installed" or "no py.typed" notes
# mypy is suppressing.

## Step 7 — what `--no-incremental` actually breaks

Setting `incremental = False` (or passing `--no-incremental`) disables both the cache write and the cache read. Several features depend on the cache to work at all:

- `--watch` / `dmypy run` — re-check on file change. Without a cache there's nothing to compare against.
- `# mypy: allow-redefinition` and dynamic class tweaks — the cross-file state that makes these annotations mean what they mean is held in the cache.
- `warn_unused_ignores` is more conservative on a cold cache (some ignores only get re-evaluated once the incremental state is rebuilt).

In short: don't use `--no-incremental` to "be safe". Use it once to debug a cache corruption, then go back to the default.

In [ ]:
# Run:  rm -rf .mypy_cache
# Run:  mypy src/                    # cold
# Run:  mypy --no-incremental src/  # also cold, but slower
# Run:  mypy src/                    # warm again
#
# If you suspect a corrupted cache entry, this is the recovery
# sequence. Don't make it part of the normal loop — the incremental
# cache is the point.

## Verify

The mental model after this walk-through:

1. `.mypy_cache/` is content-fingerprinted and invalidated by config changes, plugin changes, and mypy version changes — not by file mtime.
2. `--follow-imports` picks one of four policies for what to do with imports outside the check set; `py.typed` is the gate that decides whether `normal` follows a package at all.
3. `python_version` reshapes the stdlib stub set; switching it forces a clean re-check and may surface new errors in code that was implicitly Any-typed under an older target.
4. `--no-incremental` is a debugging tool, not a CI default. Most "is mypy broken?" issues resolve by wiping `.mypy_cache/` and re-running with the default flags.

For a CI workflow that uses all three: pin the mypy version, key the cache by mypy version + a hash of the `[tool.mypy]` block, run on each Python version in the supported matrix, and use `--follow-imports=error` only if you intend to enforce py.typed across the dependency tree.

References:

- [mypy running mypy — `--follow-imports`, `--no-incremental`, `MYPYPATH`](https://mypy.readthedocs.io/en/stable/running%5Fmypy.html)
- [mypy getting started — `reveal_type`, `python_version`, the incremental default](https://mypy.readthedocs.io/en/stable/getting_started.html)